# Creekside heating DSM — PyTorch architectures → ONNX

**Deep-learning companion** to the sklearn walkthrough. Same bootstrap features;
compare **MLP / ResMLP / 1D-CNN**, peak-weighted loss, export champion to **ONNX**.

| | |
|---|---|
| **Target** | `facility_kw` |
| **Peak metric** | HE 05–09 morning MAE |
| **Artifacts** | `ml/artifacts/heating_dsm_hourly_v1.onnx` + `_feature_meta.json` |
| **Honesty** | `BAS_BOOTSTRAP_PROXY` · **CANDIDATE** |

CLI equivalent: `python -u ml/train_heating_dsm_torch.py`.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import onnxruntime as ort

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
ML = ROOT / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import artifact_paths, bootstrap_parquet_path
from feature_compile_heating_dsm import matrix_xy, morning_peak_mask
from train_heating_dsm_torch import (
    MLP, ResMLP, HourCNN, bake_off_torch, export_onnx,
)

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device", device, "torch", torch.__version__)

## 1 · Load data

In [ ]:
pq = bootstrap_parquet_path()
if not pq.is_file():
    import subprocess
    subprocess.check_call([sys.executable, "-u", str(ML / "build_bootstrap_dataset.py")], cwd=str(ROOT))
df = pd.read_parquet(pq)
X, y, groups, cols = matrix_xy(df)
peak = morning_peak_mask(df)
print(df.shape, "n_features", len(cols))

## 2 · Architecture bake-off

Uses GroupKFold by day and peak-weighted MSE (morning hours ×2.5).

In [ ]:
# epochs=35 is a reasonable laptop default; bump for final ship
result = bake_off_torch(df, n_splits=3, epochs=35, device=device)
cv = pd.DataFrame(
    [{"family": k, **v} for k, v in result["cv"].items()]
).sort_values("mae_peak_05_09")
display(cv)
print("champion", result["champion"])

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.barh(cv["family"], cv["mae_peak_05_09"], color="#4C78A8")
ax.set_xlabel("OOF morning-peak MAE [kW]")
ax.set_title("PyTorch architecture bake-off")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.savefig(PATHS["figures"] / "torch_leaderboard.png", dpi=140, bbox_inches="tight")
plt.show()

## 3 · Export ONNX + round-trip

In [ ]:
export_onnx(result["model"], result["n_in"], PATHS["onnx"], device=device)

meta = {
    "feature_cols": result["feature_cols"],
    "scaler_mean": result["scaler"].mean_.tolist(),
    "scaler_scale": result["scaler"].scale_.tolist(),
    "champion": result["champion"],
    "cv": result["cv"],
    "schema": "creekside.heating_dsm_hourly.v1",
}
PATHS["feature_meta"].write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

Xs = result["scaler"].transform(X[:16])
with torch.no_grad():
    torch_pred = result["model"](torch.tensor(Xs, dtype=torch.float32)).numpy()
sess = ort.InferenceSession(str(PATHS["onnx"]), providers=["CPUExecutionProvider"])
onnx_pred = sess.run(None, {"features": Xs.astype(np.float32)})[0].reshape(-1)
max_abs = float(np.max(np.abs(torch_pred - onnx_pred)))
print("wrote", PATHS["onnx"])
print("wrote", PATHS["feature_meta"])
print("ONNX round-trip max |Δ|", max_abs)
assert max_abs < 1e-4, "ONNX mismatch"
pd.DataFrame({"torch": torch_pred, "onnx": onnx_pred}).head()

## 4 · Inference sketch for later sim loops

```python
import onnxruntime as ort, json, numpy as np
meta = json.loads(open("ml/artifacts/heating_dsm_hourly_v1_feature_meta.json").read())
sess = ort.InferenceSession("ml/artifacts/heating_dsm_hourly_v1.onnx")
x = (raw_features - mean) / scale   # same order as meta["feature_cols"]
kw = sess.run(None, {"features": x.astype("float32")})[0]
```

Swap bootstrap parquet for an EnergyPlus DM farm later — **do not change**
`FEATURE_COLS` without bumping the schema version.